# CNT rows of `tab:app-representation` (quoted as "reference from prior CNT runs")

Printed output only. `item2_representation_n40.json`, which this notebook was meant to write, is not in the archive.

**Needs:** DTD, the CNT checkpoint.

**Maintained runnable path:** none. This notebook (with `cnt_sam_dino_compare.ipynb`, in this same directory) is the only record of `tab:app-representation`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = os.environ.get('DATA_ROOT', "/content/drive/MyDrive/DATA_ROOT")  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
dtd_root = os.path.join(PROJECT_DIR, "dtd/images")
FORENSICS_OUT = os.path.join(PROJECT_DIR, "ot_metric_forensics")
os.makedirs(FORENSICS_OUT, exist_ok=True)

!pip install -q pot scikit-learn
print("DTD:", os.path.isdir(dtd_root))

In [ ]:
import numpy as np, torch
import matplotlib.pyplot as plt
import ot as pot
from scipy.optimize import linear_sum_assignment

rng = np.random.default_rng(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

N = 800
A = rng.multivariate_normal([0,0], [[2.0,0],[0,0.2]], size=N)
B = rng.multivariate_normal([6,6], [[0.2,0],[0,2.0]], size=N)

def linear_mid(A,B): return 0.5*A + 0.5*B
def ot_mid(A,B):
    C = pot.dist(A,B,metric="sqeuclidean")
    row,col = linear_sum_assignment(C)
    return 0.5*A[row] + 0.5*B[col]

Lm, Om = linear_mid(A,B), ot_mid(A,B)
true_mean = np.array([3.0,3.0])

fig,ax=plt.subplots(1,3,figsize=(13,4.2))
for a,(P,t) in zip(ax,[(None,"A & B"),(Lm,"LINEAR mid"),(Om,"OT mid")]):
    a.scatter(A[:,0],A[:,1],s=4,alpha=.3,c='tab:blue',label='A')
    a.scatter(B[:,0],B[:,1],s=4,alpha=.3,c='tab:red',label='B')
    if P is not None: a.scatter(P[:,0],P[:,1],s=5,c='tab:green')
    a.scatter(*true_mean,s=120,c='k',marker='x'); a.set_title(t); a.set_xlim(-4,10); a.set_ylim(-4,10)
plt.show()
print("LINEAR mid:  mean",Lm.mean(0).round(2)," std",Lm.std(0).round(2))
print("OT mid:      mean",Om.mean(0).round(2)," std",Om.std(0).round(2))
print("(both means ~[3,3]; key is LINEAR collapses std, OT preserves spread)")

In [ ]:
cA=np.array([[2.0,0],[0,0.2]]); cB=np.array([[0.2,0],[0,2.0]])
sM = 0.5*np.sqrt(cA)+0.5*np.sqrt(cB); cM=sM@sM
T_true = rng.multivariate_normal(true_mean, cM, size=N)   # analytic McCann midpoint

def w2(P,Q):
    C=pot.dist(P,Q,metric="sqeuclidean")
    return np.sqrt(pot.emd2(np.ones(len(P))/len(P), np.ones(len(Q))/len(Q), C))
def mean_dist(P,Q): return np.linalg.norm(P.mean(0)-Q.mean(0))
def cov_dist(P,Q):  return np.linalg.norm(np.cov(P.T)-np.cov(Q.T))
def energy_dist(P,Q):
    d=lambda X,Y: pot.dist(X,Y,metric="euclidean").mean()
    return 2*d(P,Q)-d(P,P)-d(Q,Q)

print("Distance of each midpoint to the TRUE McCann interpolant (lower=better):\n")
for name,fn in [("W2 (transport judge)",w2),("mean-only",mean_dist),("cov/spread",cov_dist),("energy dist",energy_dist)]:
    dl,do=fn(Lm,T_true),fn(Om,T_true)
    print(f"  {name:22s}: LIN {dl:.3f}  OT {do:.3f}  -> {'OT' if do<dl else 'LIN'} closer to truth")
print("\nHONEST metric = one that says OT closer (OT matches true interpolant).")
print("If W2 says OT but cov/energy DON'T, W2's preference may be circular.")

In [ ]:
# Circularity stress test: when A and B are SIMILAR, linear is already near-optimal.
# Honest metrics should say OT ~ LIN. Does W2 spuriously favor OT anyway?
print("Adversarial case: nearly-identical A, B (linear should already be near-perfect)\n")
for gap, shape_diff in [(0.5, 0.0), (0.5, 0.3), (3.0, 0.0)]:
    cA = [[1.0,0],[0,1.0]]
    cB = [[1.0+shape_diff,0],[0,1.0-shape_diff*0.5]]
    A2 = rng.multivariate_normal([0,0], cA, size=N)
    B2 = rng.multivariate_normal([gap,gap], cB, size=N)
    Lm2, Om2 = linear_mid(A2,B2), ot_mid(A2,B2)
    # true interpolant
    sM2 = 0.5*np.sqrt(np.array(cA))+0.5*np.sqrt(np.array(cB)); cM2=sM2@sM2
    T2 = rng.multivariate_normal(0.5*np.array([0,0])+0.5*np.array([gap,gap]), cM2, size=N)
    w_l,w_o = w2(Lm2,T2), w2(Om2,T2)
    c_l,c_o = cov_dist(Lm2,T2), cov_dist(Om2,T2)
    print(f"gap={gap}, shape_diff={shape_diff}:")
    print(f"   W2:        LIN {w_l:.3f}  OT {w_o:.3f}  -> {'OT' if w_o<w_l else 'LIN/tie'} (margin {abs(w_l-w_o):.3f})")
    print(f"   cov/spread:LIN {c_l:.3f}  OT {c_o:.3f}  -> {'OT' if c_o<c_l else 'LIN/tie'} (margin {abs(c_l-c_o):.3f})")
    print()
print("If W2 keeps a large OT margin even at shape_diff=0 (identical shapes),")
print("that's the circularity: W2 rewards OT for matching its own objective, not for truth.")

In [ ]:
# Does OT's advantage depend on whether linear's pairing is already good?
# Case A: RANDOM pairing (sets with no correspondence) - like CNT textons
# Case B: ALIGNED pairing (A[i] already corresponds to B[i]) - like grid patches
print("Testing: OT advantage vs quality of linear's existing pairing\n")

def make_aligned_pair(gap, shape_diff, n=N):
    """B is a transformed copy of A -> A[i] genuinely corresponds to B[i]."""
    base = rng.multivariate_normal([0,0],[[1,0],[0,1]],size=n)
    A = base.copy()
    # B = A shifted + reshaped (so index i DOES correspond)
    B = base * np.array([1+shape_diff, 1-shape_diff*0.5]) + np.array([gap,gap])
    return A, B

def make_random_pair(gap, shape_diff, n=N):
    """A, B independent -> index i does NOT correspond (random pairing)."""
    A = rng.multivariate_normal([0,0],[[1,0],[0,1]],size=n)
    B = rng.multivariate_normal([gap,gap],[[1+shape_diff,0],[0,1-shape_diff*0.5]],size=n)
    return A, B

for label, maker in [("ALIGNED (grid-like)", make_aligned_pair),
                     ("RANDOM (set-like/CNT)", make_random_pair)]:
    A2,B2 = maker(3.0, 0.0)
    Lm2,Om2 = linear_mid(A2,B2), ot_mid(A2,B2)
    sM2=0.5*np.sqrt(np.array([[1,0],[0,1]]))+0.5*np.sqrt(np.array([[1,0],[0,1]])); cM2=sM2@sM2
    T2 = rng.multivariate_normal([1.5,1.5], cM2, size=N)
    print(f"{label}:")
    print(f"   LIN std {Lm2.std(0).round(2)}  OT std {Om2.std(0).round(2)}  (truth std ~[1,1])")
    print(f"   cov/spread to truth: LIN {cov_dist(Lm2,T2):.3f}  OT {cov_dist(Om2,T2):.3f}  -> {'OT wins big' if cov_dist(Om2,T2)<cov_dist(Lm2,T2)*0.7 else 'similar'}")
    print()
print("PREDICTION: ALIGNED -> OT~LIN (linear pairing already good, like DINO grid)")
print("            RANDOM  -> OT wins big (linear pairing bad, like CNT textons)")
print("This would explain WHY CNT wins and DINO/SAM don't: it's about pairing quality.")

In [ ]:
# Minimal CNT load (from your pipeline). Run after Cell 1's install of pot/sklearn.
%cd /content
!rm -rf cnt-siga24
!git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24
!pip install -q -r cnt-siga24/requirements.in
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
ckpts_drive = os.path.join(PROJECT_DIR, "cnt_ckpts")
!rm -rf /content/cnt-siga24/ckpts && ln -sfn "{ckpts_drive}" /content/cnt-siga24/ckpts
%cd /content/cnt-siga24

import sys, copy, random
sys.path.insert(0, "/content/cnt-siga24")
import torch.nn.functional as F
from torch import nn
from pathlib import Path
from src.inference.core import TexAutoEncoderInference, load_image

model = TexAutoEncoderInference(ckpt_path=Path("/content/cnt-siga24/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
print("CNT loaded")

def load_texture(p): return load_image(Path(p), IMG_SIZE).to(device)
def first_img(cat):
    fs=sorted(f for f in os.listdir(os.path.join(dtd_root,cat)) if f.lower().endswith(('.jpg','.png')))
    return os.path.join(dtd_root,cat,fs[0])

def get_inputs(img):
    with torch.no_grad():
        blob = model.encode(img)
        layout = model.splat(blob)
        bf = layout["layer_level_layouts"][-1][0]["blob_features"].clone()  # (1, N, 384)
        si = layout["layer_level_layouts"][-1][0]["score_img"].clone()      # (1, N, H, W)
        tree = blob["layer_tree_levels"][-1][0]
    return blob, bf, si, tree

def style_map(bf, si):
    # per-pixel feature field: (1, 384, H, W) -> points (H*W, 384)
    sm = torch.einsum("bnc,bnhw->bchw", bf, si)
    return sm[0].reshape(sm.shape[1], -1).T   # (HW, 384)

In [ ]:
import ot as pot
from scipy.optimize import linear_sum_assignment

def cov_dist_t(P, Q):  # torch cov-distance (the calibrated independent referee)
    P,Q = P.float(), Q.float()
    cP = torch.cov(P.T); cQ = torch.cov(Q.T)
    return (cP-cQ).pow(2).mean().sqrt().item()

def lin_mid_idx(A, B):           # index-aligned linear
    return 0.5*A + 0.5*B
def ot_mid_hung(A, B, sub=2048): # OT midpoint via Hungarian on a subsample (your CNT recipe scale)
    n = A.shape[0]
    ia = torch.randperm(n)[:sub]; ib = torch.randperm(n)[:sub]
    a_,b_ = A[ia], B[ib]
    C = torch.cdist(a_,b_).pow(2).cpu().numpy()
    row,col = linear_sum_assignment(C)
    return 0.5*a_[torch.as_tensor(row,device=A.device)] + 0.5*b_[torch.as_tensor(col,device=A.device)]

def true_interp_proxy(A, B, sub=2048):
    # best available "truth" proxy: the OT (McCann) midpoint at high sample count.
    # We measure how close LIN vs OT are to THIS. (OT is favored by construction here,
    # so we ALSO report raw variance preservation vs the mean of A,B variances.)
    return ot_mid_hung(A,B,sub)

def variance_score(M, A, B):
    # honest, construction-free check: does midpoint preserve the AVG of A,B spreads?
    target_std = 0.5*(A.std(0)+B.std(0))           # what a faithful interpolant should roughly keep
    return (M.std(0)-target_std).abs().mean().item()  # lower = preserved spread better

PAIRS = [("dotted","honeycombed"),("woven","knitted"),("striped","zigzagged"),
         ("cracked","veined"),("scaly","cobwebbed"),("braided","spiralled")]

print(f"{'pair':22s} | TEXTON set (no corr.)      | PIXEL-map (pos-aligned)")
print(f"{'':22s} | LIN-var  OT-var  winner       | LIN-var  OT-var  winner")
print("-"*92)
for ca,cb in PAIRS:
    _,bfa,sia,tra = get_inputs(load_texture(first_img(ca)))
    _,bfb,sib,trb = get_inputs(load_texture(first_img(cb)))

    # (1) TEXTON SETS: unordered -> expect OT to win (RANDOM-like)
    TA = bfa[0]; TB = bfb[0]                       # (N,384) texton features, no canonical pairing
    lin_t = variance_score(lin_mid_idx(TA,TB), TA, TB)
    ot_t  = variance_score(ot_mid_hung(TA,TB, sub=min(2048,TA.shape[0])), TA, TB)

    # (2) PIXEL STYLE-MAP: position-aligned -> expect OT ~ LIN (ALIGNED-like)
    PA = style_map(bfa,sia); PB = style_map(bfb,sib)
    lin_p = variance_score(lin_mid_idx(PA,PB), PA, PB)
    ot_p  = variance_score(ot_mid_hung(PA,PB), PA, PB)

    wt = "OT" if ot_t < lin_t*0.9 else ("LIN" if lin_t<ot_t*0.9 else "~tie")
    wp = "OT" if ot_p < lin_p*0.9 else ("LIN" if lin_p<ot_p*0.9 else "~tie")
    print(f"{ca[:10]+'/'+cb[:10]:22s} | {lin_t:.3f}   {ot_t:.3f}   {wt:8s}    | {lin_p:.3f}   {ot_p:.3f}   {wp}")

In [ ]:
ca, cb = "dotted", "honeycombed"
_,bfa,sia,tra = get_inputs(load_texture(first_img(ca)))
_,bfb,sib,trb = get_inputs(load_texture(first_img(cb)))

TA, TB = bfa[0], bfb[0]
PA, PB = style_map(bfa,sia), style_map(bfb,sib)

print("TEXTON features:")
print(f"  shape {tuple(TA.shape)} | std {TA.std().item():.4f} | mean|val| {TA.abs().mean().item():.4f}")
print(f"  per-texton std range: {TA.std(1).min().item():.4f} .. {TA.std(1).max().item():.4f}")
# how many textons are basically empty?
active = (TA.abs().max(1).values > 0.01).sum().item()
print(f"  active textons (max|val|>0.01): {active}/{TA.shape[0]}")

print("\nPIXEL style-map:")
print(f"  shape {tuple(PA.shape)} | std {PA.std().item():.4f} | mean|val| {PA.abs().mean().item():.4f}")

# the actual quantities the broken metric used:
print("\nWhat variance_score saw (dotted/honeycombed, TEXTONS):")
lin = 0.5*TA+0.5*TB
target_std = 0.5*(TA.std(0)+TB.std(0))
print(f"  A std mean {TA.std(0).mean():.4f} | B std mean {TB.std(0).mean():.4f} | target {target_std.mean():.4f}")
print(f"  LIN-mid std mean {lin.std(0).mean():.4f}  (if ~= target, linear already preserves -> aligned case)")
print(f"  raw |LIN.std - target| mean: {(lin.std(0)-target_std).abs().mean():.6f}")
print(f"  --> if this is ~0, linear ALREADY preserves variance here (no collapse to detect)")

In [ ]:
def cloud_spread(P):
    """Total spread = mean squared distance from centroid. The real 'is it collapsed' measure."""
    c = P.mean(0, keepdim=True)
    return (P - c).pow(2).sum(1).mean().item()   # scalar: how spread out the cloud is

def ot_mid_hung(A, B, sub=2048):
    n = A.shape[0]
    ia = torch.randperm(n)[:min(sub,n)]; ib = torch.randperm(n)[:min(sub,n)]
    a_,b_ = A[ia], B[ib]
    C = torch.cdist(a_,b_).pow(2).cpu().numpy()
    row,col = linear_sum_assignment(C)
    return 0.5*a_[torch.as_tensor(row,device=A.device)] + 0.5*b_[torch.as_tensor(col,device=A.device)]

print(f"{'pair':22s} | TEXTON spread (×1e3)        | PIXEL spread (×1e3)")
print(f"{'':22s} | A    B    LIN   OT   collapse | A    B    LIN   OT   collapse")
print("-"*98)
for ca,cb in PAIRS:
    _,bfa,sia,_ = get_inputs(load_texture(first_img(ca)))
    _,bfb,sib,_ = get_inputs(load_texture(first_img(cb)))
    for label,(A,B) in [("T",(bfa[0],bfb[0])), ("P",(style_map(bfa,sia),style_map(bfb,sib)))]:
        sA,sB = cloud_spread(A)*1e3, cloud_spread(B)*1e3
        sL = cloud_spread(0.5*A+0.5*B)*1e3
        sO = cloud_spread(ot_mid_hung(A,B))*1e3
        avg = 0.5*(sA+sB)
        # "collapse" = how far below the avg-of-endpoints spread the midpoint sits (linear should collapse, OT shouldn't)
        if label=="T":
            tline = f"{ca[:9]+'/'+cb[:9]:22s} | {sA:.2f} {sB:.2f} {sL:.2f}  {sO:.2f}  L{sL/avg:.0%} O{sO/avg:.0%}"
        else:
            print(tline + f" | {sA:.2f} {sB:.2f} {sL:.2f}  {sO:.2f}  L{sL/avg:.0%} O{sO/avg:.0%}")
print("\nKey: 'collapse %' = midpoint spread / avg endpoint spread.")
print("Toy prediction: LINEAR << 100% (collapsed), OT ~100% (preserved), IF set-like (no correspondence).")
print("If BOTH ~100%: the data is aligned/tight -> OT can't help (DINO-like).")
print("If textons collapse under LIN but pixels don't: OT lives at the TEXTON level.")

In [ ]:
# Measure the SAME cloud-spread collapse on DINO patch grids, to complete the contrast.
# DINO patches ARE position-aligned across images -> prediction: LIN and OT both ~100% (OT can't help).
# NOTE: needs timm/transformers DINOv2. If it perturbs the CNT env, run this in a separate runtime
# and just transcribe the numbers into the table.

try:
    from transformers import AutoModel, AutoImageProcessor
    _dino = AutoModel.from_pretrained("facebook/dinov2-base").to(device).eval()
    for p in _dino.parameters(): p.requires_grad_(False)
    import torchvision.transforms as T
    _dn = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

    @torch.no_grad()
    def dino_tokens(path, size=224):
        im = load_image(Path(path), size).to(device)          # reuse CNT loader, [-1,1] or [0,1]?
        x = (im+1)/2 if im.min() < -0.1 else im                # -> [0,1]
        x = _dn(torch.nn.functional.interpolate(x, size, mode="bilinear"))
        out = _dino(pixel_values=x).last_hidden_state[0,1:]    # drop CLS -> (256, 768)
        return out
    DINO_OK = True
    print("DINO loaded; token shape:", tuple(dino_tokens(first_img("dotted")).shape))
except Exception as e:
    DINO_OK = False
    print("DINO load failed (likely env conflict). Run this cell in a fresh runtime. Error:", str(e)[:150])

In [ ]:
def collapse_row(A, B):
    sA,sB = cloud_spread(A), cloud_spread(B); avg=0.5*(sA+sB)
    sL = cloud_spread(0.5*A+0.5*B)
    sO = cloud_spread(ot_mid_hung(A,B))
    return sL/avg, sO/avg

import numpy as np
rows = {"CNT texton":[], "CNT pixel":[], "DINO patch":[]}
for ca,cb in PAIRS:
    _,bfa,sia,_ = get_inputs(load_texture(first_img(ca)))
    _,bfb,sib,_ = get_inputs(load_texture(first_img(cb)))
    rows["CNT texton"].append(collapse_row(bfa[0], bfb[0]))
    rows["CNT pixel"].append(collapse_row(style_map(bfa,sia), style_map(bfb,sib)))
    if DINO_OK:
        rows["DINO patch"].append(collapse_row(dino_tokens(first_img(ca)), dino_tokens(first_img(cb))))

print(f"{'representation':14s} | LIN keeps | OT keeps | OT advantage")
print("-"*58)
SUMMARY = {}
for k,v in rows.items():
    if not v: continue
    v=np.array(v); lin,ot = v[:,0].mean(), v[:,1].mean()
    SUMMARY[k]=(lin,ot)
    print(f"{k:14s} |   {lin:5.0%}   |   {ot:5.0%}  |   {'+{:.0%}'.format(ot-lin)}")
print("\nPrediction: CNT (set-like) -> big OT advantage; DINO (aligned) -> ~0 advantage.")
print("This is the whole thesis in one table.")